# PEMS-SF 1D-CNN 训练笔记 (V2.1.0 - 7类全周 + 梯度特征)
目标： 利用 1D-CNN 的局部特征提取能力，结合原始流量与梯度信息，解决周二/周三混淆问题。  
数据： PEMS-SF，双通道输入 (Raw Flow + Gradient)。  
实验设计： 7分类 (0-6)，无 Kalman 平滑。

# 升级版
方案一：引入 Focal Loss（专治“周三隐形”和“周五霸权”）  
原理：目前的 CrossEntropyLoss 对所有错误的惩罚是一样的。但模型发现，“把周三猜成周四”比“去学周三的特征”更容易降低 Loss。  
Focal Loss 会动态调整权重：越难分的样本（比如周三），Loss 权重越大；越好分的样本（比如周一），权重越小。 这会逼着模型去啃硬骨头。

In [ ]:
# ==========================================
# Cell 1: 基础依赖与配置
# ==========================================
import os, pathlib, json, math, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# 配置
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ROOT = pathlib.Path('.')
# 输出目录 V2.1.0
CM_DIR = ROOT / 'confusion_matrices210'
CM_DIR.mkdir(exist_ok=True)
MODEL_DIR = ROOT / 'models210' 
MODEL_DIR.mkdir(exist_ok=True)

# 随机种子 - 保证可复现性
def set_seed(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)
print(f'Device: {DEVICE}, Output Dir: {CM_DIR}')

In [ ]:
# 训练与解释的可复现性与确定性设置
torch.manual_seed(0); np.random.seed(0); random.seed(0)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
# ==========================================
# Cell 2: 数据读取与预处理 (双通道改造)
# ==========================================

# 路径配置
train_path = ROOT / 'data' / 'PEMS_train'
test_path  = ROOT / 'data' / 'PEMS_test'
labels_train_path = ROOT / 'data' / 'PEMS_trainlabels'
labels_test_path  = ROOT / 'data' / 'PEMS_testlabels'
stations_list_path = ROOT / 'data' / 'stations_list'
groups_index_path = ROOT / 'data' / 'processed' / 'pems_sf_groups_index.json' # 假设你有这个分组文件

# 基础解析函数
def parse_day_matrix(line: str):
    line = line.strip()
    if line.startswith('[') and line.endswith(']'): line = line[1:-1]
    rows = [r.strip() for r in line.split(';') if r.strip()]
    data = []
    for r in rows:
        nums = [float(x) for x in r.split() if x.replace('.', '', 1).isdigit()]
        if nums: data.append(nums)
    if not data: return None
    arr = np.full((len(data), max(len(rr) for rr in data)), np.nan, dtype=float)
    for i, rr in enumerate(data): arr[i, :len(rr)] = rr
    return arr

def iter_day_matrices(path: pathlib.Path, limit=None):
    with path.open('r', encoding='utf-8', errors='ignore') as f:
        for i, line in enumerate(f):
            if limit is not None and i >= limit: break
            mat = parse_day_matrix(line)
            yield mat if mat is not None else None

def load_labels(path: pathlib.Path):
    txt = path.read_text(encoding='utf-8', errors='ignore').strip().strip('[]')
    return np.array([int(x) for x in txt.split() if x.isdigit()], dtype=int)

# 加载元数据
labels_train = load_labels(labels_train_path)
labels_test  = load_labels(labels_test_path)
station_ids = [int(x) for x in stations_list_path.read_text(encoding='utf-8', errors='ignore').strip().strip('[]').split() if x.isdigit()]

# 加载分组掩码
group_station_masks = {}
if groups_index_path.exists():
    processed = json.loads(groups_index_path.read_text(encoding='utf-8'))
    groups_index = processed.get('groups_index', {})
    sid_to_pos = {sid: i for i, sid in enumerate(station_ids)}
    for g in ['g1','g2','g3','g4','g5']:
        mask = np.zeros(len(station_ids), dtype=bool)
        for sid in groups_index.get(g, []):
            if sid in sid_to_pos: mask[sid_to_pos[sid]] = True
        group_station_masks[g] = mask
    print('分组掩码加载完毕。')
else:
    print('警告：未找到分组文件，将使用全量数据。')
    group_station_masks = {'all': np.ones(len(station_ids), dtype=bool)}

# === 核心修改：曲线清洗与特征工程 ===
def _process_curve(curve: np.ndarray) -> np.ndarray:
    """
    输入: 原始单条曲线 (144,)
    输出: 双通道数据 (2, 144) -> [Normalized_Flow, Gradient]
    """
    # 1. 长度对齐与插值
    curve = curve[:144]
    if curve.shape[0] < 144:
        curve = np.pad(curve, (0, 144 - curve.shape[0]), mode='constant', constant_values=np.nan)
    idx = np.arange(curve.size)
    mask = ~np.isnan(curve)
    if mask.any():
        curve = np.interp(idx, idx[mask], curve[mask])
    curve = np.nan_to_num(curve, nan=0.0)

    # 2. 归一化 (Min-Max 到 0-1) - 这一步很重要，保证梯度尺度一致
    min_val, max_val = curve.min(), curve.max()
    denom = max_val - min_val + 1e-8
    norm_curve = (curve - min_val) / denom
    
    # 3. 计算梯度 (Gradient/Difference) - 捕捉形状变化
    # np.gradient 计算中心差分，形状保持 (144,)
    grad_curve = np.gradient(norm_curve)
    
    # 4. 堆叠通道 (Channel First for PyTorch CNN: C, L)
    # output shape: (2, 144)
    result = np.stack([norm_curve, grad_curve], axis=0)
    
    return result.astype(np.float32)

# Dataset 定义
class PEMS_CNNDataset(Dataset):
    def __init__(self, split_path: pathlib.Path, labels: np.ndarray, use_mask: np.ndarray):
        self.samples = []
        
        for day_i, mat in enumerate(iter_day_matrices(split_path)):
            if day_i >= len(labels): break
            if mat is None or mat.shape[1] < 144: continue
            
            original_label = int(labels[day_i])
            # 7分类：原始标签 1-7 -> 映射为 0-6
            # 1:Mon(0), 2:Tue(1), 3:Wed(2), 4:Thu(3), 5:Fri(4), 6:Sat(5), 7:Sun(6)
            y = original_label - 1 
            if y < 0 or y > 6: continue 
            
            # 应用分组掩码
            if use_mask.shape[0] == mat.shape[0]:
                mat = mat[use_mask, :]
            
            for sidx in range(mat.shape[0]):
                # 获取双通道数据 (2, 144)
                two_channel_seq = _process_curve(mat[sidx, :144])
                
                if not np.isfinite(two_channel_seq).all(): continue
                self.samples.append((two_channel_seq, y))
        
        self.n = len(self.samples)
    
    def __len__(self): return self.n
    def __getitem__(self, idx):
        seq, y = self.samples[idx]
        return torch.from_numpy(seq), torch.tensor(y, dtype=torch.long)

In [ ]:
# ==========================================
# Cell 3: 构建DataLoader
# ==========================================
group_loaders = {}
for g, mask in group_station_masks.items():
    print(f'准备分组 {g} 数据...')
    # Train
    ds_train_full = PEMS_CNNDataset(train_path, labels_train, mask)
    # 80/20 拆分训练验证
    n_train = int(len(ds_train_full) * 0.8)
    n_val = len(ds_train_full) - n_train
    train_subset, val_subset = random_split(ds_train_full, [n_train, n_val], 
                                            generator=torch.Generator().manual_seed(42))
    
    # Test
    ds_test = PEMS_CNNDataset(test_path, labels_test, mask)
    
    group_loaders[g] = {
        'train': DataLoader(train_subset, batch_size=64, shuffle=True, num_workers=0),
        'val':   DataLoader(val_subset, batch_size=128, shuffle=False, num_workers=0),
        'test':  DataLoader(ds_test, batch_size=128, shuffle=False, num_workers=0)
    }
    print(f'  - Train: {len(train_subset)}, Val: {len(val_subset)}, Test: {len(ds_test)}')

## 模型定义：1D-CNN Fusion Hybrid
交通流解释：1D-CNN 擅长通过卷积核捕捉局部平移不变的波形特征（如拥堵的起始坡度），并相比 LSTM 训练效率更高。结合 5D 静态特征增强对整体结构的判别力；卡尔曼滤波作为前置/后置平滑去噪。

In [ ]:
# ==========================================
# Cell 4: 1D-CNN 模型定义
# ==========================================
class Simple1DCNN(nn.Module):
    def __init__(self, in_channels=2, num_classes=7):
        super(Simple1DCNN, self).__init__()
        
        # Block 1: 捕捉大尺度特征 (Kernel Size = 7)
        self.block1 = nn.Sequential(
            nn.Conv1d(in_channels, 32, kernel_size=7, padding=3),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2) # 144 -> 72
        )
        
        # Block 2: 捕捉中尺度特征 (Kernel Size = 5)
        self.block2 = nn.Sequential(
            nn.Conv1d(32, 64, kernel_size=5, padding=2),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2) # 72 -> 36
        )
        
        # Block 3: 捕捉微观细节/噪声 (Kernel Size = 3)
        self.block3 = nn.Sequential(
            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            # 不再池化，保留更多时间信息供 averaging
        )
        
        # Global Average Pooling: 无论时间多长，都压扁成 Channel 数
        self.gap = nn.AdaptiveAvgPool1d(1)
        
        # Classification Head
        self.fc = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes)
        )
        
    def forward(self, x):
        # x shape: (Batch, 2, 144)
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        
        x = self.gap(x)       # (Batch, 128, 1)
        x = x.view(x.size(0), -1) # (Batch, 128)
        logits = self.fc(x)
        return logits

## 多组训练循环与评估
- 对每个分组分别训练模型，使用验证集评估。
- 不再进行参数加权平均，而是基于分组路由的 MoE 评估逻辑：测试时，根据数据属于哪个组，调用该组对应的专家模型进行预测。

In [ ]:
# 1. 定义 Focal Loss 类 (加在 Cell 5 前面)
class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction
        self.ce = nn.CrossEntropyLoss(reduction='none', label_smoothing=0.1) # 保留一点 smoothing

    def forward(self, inputs, targets):
        logpt = -self.ce(inputs, targets)
        pt = torch.exp(logpt)
        # 核心公式：(1-pt)^gamma 会让高置信度的样本权重几乎为0
        loss = self.alpha * (1 - pt) ** self.gamma * -logpt
        if self.reduction == 'mean': return loss.mean()
        return loss.sum()

In [ ]:
# ==========================================
# Cell 5: 训练循环
# ==========================================
def train_model(loaders, group_name, epochs=30):
    model = Simple1DCNN(in_channels=2, num_classes=7).to(DEVICE)
    # 使用 AdamW 优化器，稍加 Weight Decay 防止过拟合
    optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
    # Label Smoothing: 稍微容忍一点不确定性，有助于相似类别的区分
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    
    # gamma=2 是经验值，专注于难分样本
    criterion = FocalLoss(gamma=2.0).to(DEVICE)

    best_acc = 0.0
    best_state = None
    
    train_loader = loaders['train']
    val_loader = loaders['val']
    
    for epoch in range(1, epochs+1):
        model.train()
        total_loss = 0
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            out = model(x)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            
        # Validation
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(DEVICE), y.to(DEVICE)
                out = model(x)
                preds = torch.argmax(out, dim=1)
                correct += (preds == y).sum().item()
                total += y.size(0)
        
        val_acc = correct / total if total > 0 else 0
        
        if val_acc > best_acc:
            best_acc = val_acc
            best_state = model.state_dict()
            
        if epoch % 5 == 0:
            print(f"[{group_name}] Epoch {epoch}/{epochs} | Loss: {total_loss/len(train_loader):.4f} | Val Acc: {val_acc:.2%}")
            
    return best_state, best_acc

# 开始训练所有组
group_models = {}
for g, loaders in group_loaders.items():
    print(f"\n>>> 开始训练分组: {g}")
    state, acc = train_model(loaders, g, epochs=50) # 增加 Epoch，CNN 收敛可能慢一点
    group_models[g] = state
    print(f"FAILED" if state is None else f"[{g}] 训练完成，最佳验证准确率: {acc:.2%}")
    if state:
        torch.save(state, MODEL_DIR / f'cnn_v210_{g}.pth')

print("\n所有模型训练结束。")

## SHAP 解释性分析（DeepExplainer）
交通流解释：量化 5D 静态特征对某个预测结果的边际贡献，如早高峰强度对工作日识别的作用。

In [ ]:
class MLPForSHAP(nn.Module):
    # 修改 num_classes 默认为 4
    def __init__(self, fusion_dim=69, num_classes=4, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(fusion_dim, 128), nn.ReLU(), nn.Dropout(dropout), nn.Linear(128, num_classes))
    
    def forward(self, x):
        return self.net(x)

# 确保推理模式，避免 Dropout 引入随机性导致 SHAP 可加性检查失败
# 修改：使用 G1 (通勤组) 的专家模型进行 SHAP 分析，因为 G1 最具代表性
mlp_shap = MLPForSHAP(fusion_dim=69).to(DEVICE)
mlp_shap.eval()

target_group = 'g1'
expert_model = CNNKalmanClassifier().to(DEVICE)
expert_state_path = ROOT / f'model_cnn_kalman_{target_group}_v200.pth'

if expert_state_path.exists():
    expert_model.load_state_dict(torch.load(expert_state_path, map_location=DEVICE))
    print(f'SHAP 分析：已加载 {target_group} 专家模型')
else:
    print(f'警告：未找到 {target_group} 模型，SHAP 将基于随机初始化权重。')

# 将专家模型的 MLP 权重复制到 SHAP MLP
mlp_layers = list(expert_model.mlp.children()); mlp_shap_layers = list(mlp_shap.net.children())
for i in range(len(mlp_layers)):
    if isinstance(mlp_layers[i], nn.Linear) and isinstance(mlp_shap_layers[i], nn.Linear):
        mlp_shap_layers[i].weight.data = mlp_layers[i].weight.data.detach().clone()
        mlp_shap_layers[i].bias.data = mlp_layers[i].bias.data.detach().clone()

# 背景样本（来自 g1 训练加载器）
bg = None
if target_group in group_loaders:
    # 使用目标组的数据作为背景
    train_loader = group_loaders[target_group][0]
    for i, (seq, f5, y) in enumerate(train_loader):
        if i >= 2: break
        with torch.no_grad():
            seq = seq.to(DEVICE).float(); f5 = f5.to(DEVICE).float()
            # 使用 expert_model 的 CNN
            x = seq.permute(0, 2, 1)
            x = expert_model.cnn(x)
            x = x.squeeze(-1)
            fusion = torch.cat([x, f5], dim=1)
            bg = fusion if bg is None else torch.cat([bg, fusion], dim=0)
else:
    # Fallback
    for g, (train_loader, _) in group_loaders.items():
        break 

if bg is None:
    bg = torch.zeros((32, 69), device=DEVICE)
bg = torch.nan_to_num(bg, nan=0.0, posinf=1.0, neginf=0.0)

# 选择一个分组的小批次做解释
# 修改：从目标组取样
if target_group in group_loaders:
    seq_b, f5_b, y_b = next(iter(group_loaders[target_group][0]))
else:
    seq_b, f5_b, y_b = next(iter(next(iter(group_loaders.values()))[0]))

seq_b = seq_b.to(DEVICE).float(); f5_b = f5_b.to(DEVICE).float()
with torch.no_grad():
    x = seq_b.permute(0, 2, 1)
    x = expert_model.cnn(x)
    x = x.squeeze(-1)
    fusion_b = torch.cat([x, f5_b], dim=1)
fusion_b = torch.nan_to_num(fusion_b, nan=0.0, posinf=1.0, neginf=0.0)

try:
    explainer = shap.DeepExplainer(mlp_shap, bg)
    shap_values = explainer.shap_values(fusion_b, check_additivity=False)
except Exception as e:
    print('DeepExplainer 失败，回退 GradientExplainer:', str(e))
    explainer = shap.GradientExplainer(mlp_shap, bg)
    shap_values = explainer.shap_values(fusion_b)

# 仅输出 5D 特征维度的平均绝对贡献（无图）
feat_contrib = []
for c in range(4): # 修改 range(7) 为 range(4)
    sv = shap_values[c]  # (B, 69)
    f5_sv = np.abs(sv[:, -5:]).mean(axis=0)  # 取后 5 维
    feat_contrib.append(f5_sv.tolist())
print('5D 特征对各类的平均绝对贡献（样本批次）:', json.dumps(feat_contrib, ensure_ascii=False))

## 评估与阈值
混淆矩阵

In [ ]:
# ==========================================
# Cell 6: 可视化 - 组合混淆矩阵 (核心需求)
# ==========================================

def get_predictions(model_state, loader):
    """辅助函数：获取真实标签和预测标签"""
    model = Simple1DCNN(in_channels=2, num_classes=7).to(DEVICE)
    model.load_state_dict(model_state)
    model.eval()
    
    y_true = []
    y_pred = []
    
    with torch.no_grad():
        for x, y in loader:
            x = x.to(DEVICE)
            preds = torch.argmax(model(x), dim=1)
            y_true.extend(y.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())
            
    return np.array(y_true), np.array(y_pred)

def plot_combined_matrix(mode='test'):
    """
    mode: 'test' 或 'val'
    自动遍历 G1-G5，并计算 Global，画在一张图上 (2x3)
    """
    print(f"正在生成 {mode} 集混淆矩阵组合图...")
    
    # 定义子图位置：G1-G5 占前5个，Global 占第6个
    layout = {
        'g1': (0, 0), 'g2': (0, 1), 'g3': (0, 2),
        'g4': (1, 0), 'g5': (1, 1), 'global': (1, 2)
    }
    
    fig, axes = plt.subplots(2, 3, figsize=(20, 14))
    labels = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
    
    global_true = []
    global_pred = []
    
    # 遍历 G1-G5
    for g in ['g1', 'g2', 'g3', 'g4', 'g5']:
        if g not in group_models: continue
        
        # 获取预测
        loader = group_loaders[g][mode]
        y_t, y_p = get_predictions(group_models[g], loader)
        
        # 收集 Global 数据
        global_true.extend(y_t)
        global_pred.extend(y_p)
        
        # 绘图
        cm = confusion_matrix(y_t, y_p)
        # 按行归一化
        cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        
        row, col = layout[g]
        ax = axes[row, col]
        sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Blues',
                    xticklabels=labels, yticklabels=labels, cbar=False, ax=ax)
        ax.set_title(f"{g.upper()} ({mode.capitalize()})", fontsize=14, fontweight='bold')
        ax.set_ylabel('True')
        ax.set_xlabel('Predicted')

    # 绘制 Global
    cm_glob = confusion_matrix(global_true, global_pred)
    cm_glob_norm = cm_glob.astype('float') / cm_glob.sum(axis=1)[:, np.newaxis]
    
    row, col = layout['global']
    ax = axes[row, col]
    sns.heatmap(cm_glob_norm, annot=True, fmt='.2%', cmap='Blues',
                xticklabels=labels, yticklabels=labels, cbar=True, ax=ax)
    ax.set_title(f"Global All Groups ({mode.capitalize()})", fontsize=14, fontweight='bold')
    ax.set_ylabel('True')
    ax.set_xlabel('Predicted')
    
    # 保存
    plt.suptitle(f'{mode.capitalize()} Set Confusion Matrices (1D-CNN, 7-Class, V2.1.0)', 
                 fontsize=18, fontweight='bold', y=0.98)
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    save_path = CM_DIR / f'combined_{mode}_matrices_v210.png'
    plt.savefig(save_path, dpi=300)
    print(f"图表已保存: {save_path}")
    plt.close()

# 执行绘图
plot_combined_matrix('val')
plot_combined_matrix('test')